# Phase 2 unified Drive smoke test
Read-only: consumes existing IBKR, TXF and Taiwan FINAL GATE artifacts. It does not rerun historical QA or write Drive data.

In [ ]:
from pathlib import Path
if not Path('/content/drive/MyDrive').is_dir():
    from google.colab import drive
    drive.mount('/content/drive')
CODE_SHA = '0d2f6dfd99a87eb28c1701ca81c9ca723acf0216'
!pip -q install --force-reinstall --no-deps --no-cache-dir 'git+https://github.com/hh4832/market-research-core.git@0d2f6dfd99a87eb28c1701ca81c9ca723acf0216'

In [ ]:
from pathlib import Path
from market_research_core import BacktestAssumptions, BacktestEngine, MarketDataClient, MarketDataConfig, UnifiedBars

ibkr_dir = Path('/content/drive/MyDrive/00Quant_Research/data/00_Quant_Research_Hub/20261101_205431_06359faf7ef4_IBKR_FINAL_GATE_FULL')
txf_dir = Path('/content/drive/MyDrive/00Quant_Research/fubon_futures_intraday_data_validation/20261014_213932_404db609cf13')
data_root = Path('/content/drive/MyDrive/00Quant_Research/data')
assert ibkr_dir.is_dir(), ibkr_dir
assert txf_dir.is_dir(), txf_dir
assert (data_root / '_tw-market-data-lake').is_dir(), data_root
client = MarketDataClient(MarketDataConfig(ibkr_run_dir=ibkr_dir, txf_run_dir=txf_dir, taiwan_lake_root=data_root))
print('Code SHA:', CODE_SHA)
print('Pinned roots:', ibkr_dir, txf_dir, data_root, sep='\n')

In [ ]:
ibkr = client.load(market='IBKR', symbol='TSM')
txf = client.load(market='TXF', symbol='TXF')
tw = client.load(market='TW', symbol='2330', usage='BACKTEST')
for name, loaded in [('IBKR', ibkr), ('TXF', txf), ('TW', tw)]:
    assert len(loaded.frame) > 20
    assert {'data_exists','research_eligible','backtest_eligible','execution_eligible','available_at'} <= set(loaded.frame.columns)
    probe = UnifiedBars(loaded.frame.tail(2000).copy(), loaded.provenance, loaded.native_metadata)
    result = BacktestEngine().run_sma(probe, BacktestAssumptions(fast_window=5, slow_window=20))
    assert result.provenance['performance_status'] == 'RESEARCH_INTEGRATION_DEMO'
    print(name, 'PASS', 'rows=', len(loaded.frame), 'trades=', result.trade_count, 'snapshot=', loaded.provenance.snapshot_id)
print('PHASE 2 UNIFIED DRIVE SMOKE: PASS')